# Storage, Volumes & Bind Mounts

## What's covered

- **Why the container filesystem isn't enough** — the writable layer is fast, but ephemeral
- **The three mount types** — `volume`, `bind`, and `tmpfs`, and when to reach for each
- **Named volumes** — `docker volume create`, lifecycle, `-v name:/path` syntax
- **Anonymous volumes and `VOLUME` in Dockerfile** — the implicit volumes you didn't ask for
- **Bind mounts** — host paths into containers, the local-dev workhorse, the gotchas
- **`tmpfs`** — RAM-backed mounts for caches, secrets, and scratch
- **`-v` short form vs `--mount` long form** — the modern preferred syntax
- **Volume drivers** — `local` (default), `nfs`, third-party plugins
- **Sharing volumes** between containers — `--volumes-from`, multiple readers/writers
- **The Mac/Windows file-sync caveat** — why volumes are way faster than bind mounts on Docker Desktop
- **Backup, restore, migrate** — moving volume data with `tar` and helper containers
- **UID/GID gotchas** — permission-denied errors and the standard fixes
- **Where data actually lives** — `/var/lib/docker/volumes/` and the on-disk layout

By the end you should know exactly which mount type to use for any scenario, how to back up a database volume, and why your Mac is slow when you bind-mount your source tree.

## Why the container filesystem isn't enough

Recall the layer stack from notebook 02:

```
+-----------------------------+
|  writable container layer   |   <- ephemeral; deleted when container is removed
+-----------------------------+
|  read-only image layers     |
+-----------------------------+
```

When the container's process writes to its own filesystem — say, a Postgres container writing to `/var/lib/postgresql/data` — those writes land in the writable layer. That works, but it has three problems:

1. **It's ephemeral.** `docker rm` deletes the container, and the writable layer goes with it. Your database disappears.
2. **It's not shareable.** Two containers can't see each other's writable layers. So you can't easily have a sidecar reading another container's logs.
3. **It's slow for heavy writes.** The union filesystem (overlay2) does copy-up on first write, which is fine for occasional file changes but expensive for write-heavy workloads like databases.

The fix: take a path inside the container and **mount** something else there — something that lives outside the container's writable layer, and survives the container's lifecycle.

Docker has three flavours of mount.

## The three mount types

| Type | Lives where | Lifecycle | Best for |
|---|---|---|---|
| **Volume** | Docker-managed area (`/var/lib/docker/volumes/`) | Independent of any container; you delete explicitly with `docker volume rm` | **Persistent app data** — DB files, app uploads, anything that should outlive a container |
| **Bind mount** | An arbitrary path on the host filesystem | The host path's lifecycle (Docker never touches it) | **Local development** (mount your source code in), config files, host log forwarding |
| **`tmpfs`** | In RAM, on the host | Container only; gone when the container stops | **Sensitive scratch** (don't write secrets to disk), high-throughput caches |

The mental rule:

> **Use a volume by default. Use a bind mount when you need a specific path on the host. Use `tmpfs` when the data should never hit disk.**

The rest of this notebook unpacks each one.

## Named volumes — the default choice

A **named volume** is a piece of storage managed entirely by Docker. You give it a name, Docker decides where on disk it lives, and you mount it into containers by name.

### Create, list, inspect, remove

```bash
docker volume create pgdata
docker volume ls
docker volume inspect pgdata
docker volume rm pgdata
```

`docker volume inspect` shows the mountpoint on the host (e.g. `/var/lib/docker/volumes/pgdata/_data`), the driver (`local` by default), labels, and creation date.

### Mount it into a container

Short form with `-v`:

```bash
docker run -d --name db \
  -v pgdata:/var/lib/postgresql/data \
  -e POSTGRES_PASSWORD=secret \
  postgres:16
```

The `-v` syntax `<source>:<dest>` reads "mount the named volume `pgdata` at `/var/lib/postgresql/data` inside the container." Docker auto-creates `pgdata` if it doesn't exist yet — that's a convenient default, but the explicit `docker volume create` is what production scripts do.

`docker rm db` later doesn't touch `pgdata`. Spin up a new Postgres container with the same mount and your data is still there.

### When volumes mount over a non-empty container path

A gotcha worth knowing: if the container path *already had files* in the image (e.g. `/var/lib/postgresql/data` has the empty Postgres data dir baked in), and the volume is **empty on first mount**, Docker copies the image's contents into the volume before mounting. Subsequent mounts don't re-copy.

This is what lets `docker run -v pgdata:/var/lib/postgresql/data postgres:16` "just work" — first run seeds the volume with the freshly initialised data dir; later runs preserve whatever Postgres has written.

It does *not* happen for bind mounts. A bind mount unconditionally hides whatever was at the container path.

In [ ]:
# Create a volume, mount it, write something, kill the container, remount, see the data
!docker volume create demo-data
!docker run --rm -v demo-data:/data alpine:3.20 sh -c 'echo "persisted line" > /data/notes.txt'
!echo '--- container is gone but volume lives on ---'
!docker volume ls --filter name=demo-data
!echo '--- mount again in a new container ---'
!docker run --rm -v demo-data:/data alpine:3.20 cat /data/notes.txt
!docker volume rm demo-data

## Anonymous volumes and `VOLUME` in Dockerfile

Two ways to get a volume *without explicitly naming it*. Both can surprise you.

### Anonymous volumes from `-v`

If you write `-v /container/path` (just a container path, no source), Docker creates an **anonymous volume** — a volume with a random hex name — and mounts it there.

```bash
docker run -d -v /var/lib/mysql mysql:8
docker volume ls   # you'll see something like 9f6a3b... (random)
```

These accumulate forever unless you use `docker run --rm` (which deletes anonymous volumes with the container) or `docker volume prune` (which removes anonymous volumes not currently in use).

### `VOLUME` instruction in a Dockerfile

The `VOLUME` Dockerfile instruction declares that a path *will be* a volume:

```dockerfile
FROM postgres:16
# (this is already in the postgres image)
VOLUME /var/lib/postgresql/data
```

When you `docker run` this image **without** specifying a mount for that path, Docker creates an anonymous volume and mounts it there automatically. The intent is "this path must not be on the writable layer; data here should persist." It's how `postgres`, `mysql`, `mongo`, and most database official images protect users from accidentally putting their database on the throwaway writable layer.

Two gotchas:

1. **You'll accrue orphan anonymous volumes** if you don't pass `-v name:...` for these paths *and* don't use `--rm`. `docker volume prune` is the cleanup.
2. **`VOLUME` is sticky** — once declared in an image, you can't easily un-declare it in a derived image. Avoid `VOLUME` in your own Dockerfiles unless you genuinely need it; let the user (or Compose file) decide on mounts.

## Bind mounts — host paths into containers

A **bind mount** maps a path on your host directly into a container. No Docker-managed storage, no copying — the container sees the host's bytes.

```bash
docker run -d --name web \
  -v /home/me/projects/site:/usr/share/nginx/html:ro \
  nginx:alpine
```

This serves `/home/me/projects/site` from the host as Nginx's web root. Edit a file on the host, refresh the browser, you see the change. The `:ro` makes the mount read-only inside the container (a good default — the web server shouldn't be writing to your source tree).

### The classic dev workflow

Bind mounts are the local-development workhorse. The pattern:

1. Have your source code on the host (where your editor and git live).
2. Run the app inside a container with the source bind-mounted.
3. Either restart the container on each change, or use a hot-reloading runtime (`nodemon`, `flask --reload`, `uvicorn --reload`).

Trade-off: bind mounts hardcode the host path. The same `docker run` command won't work on a teammate's machine unless their paths match — which is why production deploys use volumes, not binds, and Compose files use *relative* paths that resolve against the project root.

### Gotchas

- **No image-content copy.** Unlike volumes, a bind mount silently masks whatever was at the container path. Mount your empty `./src` over `/app` and `/app`'s previous contents become invisible to the container.
- **The host path must exist.** If `/home/me/projects/site` doesn't exist, Docker creates it as an empty directory (on Linux). That's almost never what you wanted — typo in path → mysteriously empty container.
- **Permissions can confuse.** The container sees the host's UIDs and GIDs literally. We dig into this below.
- **Performance on macOS / Windows.** Docker Desktop has to bridge host filesystem events into a Linux VM. Bind mounts of large directories (think a `node_modules` with 200k files) can be brutally slow. The fix is usually a named volume for `node_modules` *plus* a bind mount for source — see the volume-driver section.

## `tmpfs` — RAM-backed mounts

A `tmpfs` mount is a chunk of host RAM that the container sees as a filesystem at a chosen path. Nothing on disk; data vanishes when the container stops.

```bash
docker run --tmpfs /tmp:size=64m,mode=1777 alpine sh
```

What it's good for:

- **High-throughput scratch.** Build steps, compilers, Redis dumps — anything that wants a fast filesystem and doesn't need to survive.
- **Secrets you don't want on disk.** Mount `tmpfs` at `/run/secrets` and write tokens there; the host's disk never touches them.
- **Read-only containers that need *some* writable space.** Pair `--read-only` (which makes the rootfs read-only) with `--tmpfs /tmp --tmpfs /var/run` so processes that insist on writing to specific paths have somewhere to go.

`--tmpfs` is short syntax. The long form via `--mount type=tmpfs,...` accepts more options.

## `-v` short form vs `--mount` long form

Docker has two ways to spell a mount. They mostly do the same thing.

### `-v` (short, terse)

```
-v <source>:<destination>[:<options>]
```

- Source is a volume name, an absolute host path (= bind mount), or omitted (= anonymous volume).
- Options is a comma-separated list: `ro`, `rw`, `z`, `Z`, `cached`, `delegated`, etc.

Examples:

```bash
-v pgdata:/var/lib/postgresql/data
-v /host/src:/app/src:ro
-v /etc/letsencrypt:/etc/letsencrypt:ro
```

### `--mount` (long, explicit)

```
--mount type=<volume|bind|tmpfs>,source=...,target=...,readonly,...
```

Examples:

```bash
--mount type=volume,source=pgdata,target=/var/lib/postgresql/data
--mount type=bind,source=/host/src,target=/app/src,readonly
--mount type=tmpfs,target=/scratch,tmpfs-size=64m
```

The trade-offs:

- **`-v` is shorter** but ambiguous: `-v foo:/data` is a volume named `foo`; `-v /foo:/data` is a bind mount of `/foo`. A typo turns one into the other and you don't notice until data goes the wrong place.
- **`--mount` is verbose** but unambiguous: you say `type=volume` or `type=bind` explicitly. The Docker docs recommend `--mount` for new code.

In Compose files (notebook 06), you'll see both: short `volumes:` strings (`-v` style) and long `volumes:` mappings (`--mount` style). Use whichever your team has already standardised on; lean to `--mount` for scripts and CI.

In [ ]:
# Demonstrate both styles producing the same effect
!docker volume create style-demo
!docker run --rm -v style-demo:/data alpine:3.20 sh -c 'echo "via -v" > /data/from-v'
!docker run --rm --mount type=volume,source=style-demo,target=/data alpine:3.20 sh -c 'echo "via --mount" > /data/from-mount; ls /data'
!echo '--- both files live in the same volume ---'
!docker run --rm -v style-demo:/data alpine:3.20 cat /data/from-v /data/from-mount
!docker volume rm style-demo

## Volume drivers

Every volume has a **driver** — the plugin that actually stores the bytes. The default is `local`, which means "a directory on this host's filesystem at `/var/lib/docker/volumes/<name>/_data`."

You can pick a different driver at creation time:

```bash
docker volume create --driver local --opt type=nfs --opt o=addr=192.168.1.10,rw \
  --opt device=:/exports/data myvol
```

The most common non-default driver is **`local` with options** for NFS or tmpfs, which lets you mount a network share or a RAM disk as a Docker volume. Third-party plugins (`docker plugin install ...`) extend the list to:

- Cloud block storage (AWS EBS via `rexray/ebs`, GCP PD)
- Distributed filesystems (GlusterFS, Ceph)
- Object storage shims (S3 via `s3fs` as a plugin)

For a single-host Docker installation, the default `local` driver is what 95% of teams use. Drivers really matter when you get to multi-host orchestrators — Kubernetes has its own CSI ecosystem; Swarm uses Docker volume drivers directly to share state across nodes.

## Sharing volumes between containers

Two ways to make multiple containers see the same data.

### Mount the same volume in each (the modern way)

```bash
docker volume create shared
docker run -d --name producer -v shared:/out alpine sh -c 'while true; do date >> /out/log; sleep 1; done'
docker run --rm -v shared:/in alpine cat /in/log
```

The volume is the source of truth; each container mounts it where it likes. Standard, recommended.

### `--volumes-from` (legacy)

```bash
docker run -d --name data alpine sh -c 'sleep infinity'
docker run --volumes-from data -v shared:/out ...
```

`--volumes-from CONTAINER` copies *all* of `CONTAINER`'s mounts into the new container at the same paths. Used a lot in the old "data container" pattern, where you'd create a stopped container *just* to own a volume. Today, the volume itself is a first-class object, so `--volumes-from` is mostly a relic. Recognise it in legacy Compose/Swarm setups; don't reach for it in new code.

### Concurrent writers — be careful

Two containers writing to the same volume at the same time can corrupt data unless the *application layer* coordinates. Postgres won't let you point two `postgres` containers at the same data dir; SQLite has well-known issues with multiple writers. Volumes don't add locking — they're just shared bytes.

## The Mac / Windows file-sync caveat

On Linux, containers and the host share the kernel, so bind mounts are essentially free — the container sees the host's inode directly. On **macOS** and **Windows**, Docker runs a hidden Linux VM, and the host's filesystem has to be bridged into it.

That bridge is slow. A `npm install` against `node_modules` bind-mounted from macOS can be **5-10x slower** than the same on a native Linux host. Same story for any directory with thousands of small files.

Three mitigations, in order of preference:

1. **Use volumes for hot directories.** Put `node_modules`, `.cache/pip`, `target/` (Rust), etc. in *named volumes*, not bind mounts. Bind-mount only the source files you actually edit. This is the most common production-quality pattern.
2. **Use the `cached` / `delegated` consistency options** on bind mounts (`-v ./src:/app/src:cached`). These relax read-after-write consistency between host and container for a big speed win. `cached` favours host writes; `delegated` favours container writes. (On modern Docker Desktop with `gRPC FUSE` or VirtioFS, these are partially deprecated but still accepted.)
3. **Use `mutagen-compose` or `docker-sync`** for one-way replication between host and a fast in-VM directory. Heavier setup; only worth it for very large source trees.

If you take nothing else from this section: **don't bind-mount `node_modules` on a Mac.** Use a named volume.

## Backup, restore, migrate

Volumes are owned by the daemon, so you can't just `cp -r` them — they're not always in a stable location and on Docker Desktop they're inside the VM. The standard pattern is a **helper container**.

### Backup a volume to a tarball on the host

```bash
docker run --rm \
  -v pgdata:/data:ro \
  -v $(pwd):/backup \
  alpine \
  tar czf /backup/pgdata.tar.gz -C /data .
```

Read the volume read-only on `/data`, bind-mount the current directory on `/backup`, tar the volume contents to `/backup/pgdata.tar.gz`. Result: `pgdata.tar.gz` in your shell's working directory.

### Restore from the tarball

```bash
docker volume create pgdata-restored
docker run --rm \
  -v pgdata-restored:/data \
  -v $(pwd):/backup \
  alpine \
  tar xzf /backup/pgdata.tar.gz -C /data
```

Then `docker run -v pgdata-restored:/var/lib/postgresql/data postgres:16` to use it.

### Migrate between hosts

Backup to tar on host A, `scp` the tar to host B, restore on host B. The same pattern works for moving a volume between Docker Desktop and a Linux server.

### Database-aware backups

For production databases, **don't tar live data files** — you can catch them mid-write and corrupt the backup. Use the database's own dump tool:

```bash
docker exec db pg_dump -U postgres mydb > mydb.sql
```

Tar of the volume is fine for *stopped* databases or for non-database data.

## UID / GID gotchas

The single most common bind-mount frustration: "I bind-mounted my project, the container can't write to it" or "the files the container creates are owned by root on my host."

### Why

A container's processes run with some UID inside the namespace. By default that UID is whatever the image's `USER` sets — often `0` (root), or sometimes a specific UID like `1000` or `999`.

When that process writes to a bind-mounted host path, **the host sees the write as being done by that same numeric UID** (because by default Docker doesn't remap UIDs — user-namespace remapping is opt-in; see notebook 08).

So if your container runs as UID 0 (root) and writes `/app/output.txt`, the file on your host is owned by `root:root`. If it runs as UID 999 (which is `postgres` inside the postgres image), the file's owned by `999:999` on the host — likely some random local user with that UID, or nobody.

### The standard fixes

- **Match UIDs.** Either build the image to run as your host UID (`docker run -u "$(id -u):$(id -g)"`) or have the image's user already match your host. This is what dev-container scripts often do.
- **Pre-create the host directory** with permissive ownership (`mkdir -p ./data && chmod 777 ./data`) for dev environments. Not for production.
- **For volumes (not bind mounts), let Docker manage UIDs.** Volume directories are owned by `root` on the host but the container's user can usually still chown/chmod them on first mount via the image's entrypoint script. This is why most database images "just work" with volumes but fight you when you bind-mount.
- **Use Docker Desktop's auto-UID translation.** On macOS/Windows, Docker Desktop transparently maps host owner to container owner for bind mounts. This is one reason bind-mounting on Desktop "feels normal" — but the abstraction leaks under load.

In [ ]:
# UID gotcha demo: container as root writes to a bind mount
!mkdir -p /tmp/uid-demo && rm -rf /tmp/uid-demo/* && ls -la /tmp/uid-demo
!echo '--- write a file from an alpine container (runs as root by default) ---'
!docker run --rm -v /tmp/uid-demo:/work alpine:3.20 sh -c 'echo hi > /work/file-from-container; id'
!ls -la /tmp/uid-demo
!echo '--- now run as our host UID/GID and see the difference ---'
!docker run --rm -u "$(id -u):$(id -g)" -v /tmp/uid-demo:/work alpine:3.20 sh -c 'echo hi2 > /work/file-as-me; id'
!ls -la /tmp/uid-demo
!rm -rf /tmp/uid-demo

## Where data actually lives

For curiosity and for troubleshooting, here's the on-disk layout on a typical Linux Docker installation.

```
/var/lib/docker/
├── overlay2/              # image and container layer data
│   ├── <layer-id>/
│   │   ├── diff/          # the actual files for this layer
│   │   ├── link
│   │   ├── lower          # pointers to parent layers
│   │   └── work/
│   └── l/                 # symlink shorthand for layer paths
│
├── volumes/               # named volumes
│   ├── pgdata/
│   │   ├── _data/         # the volume's contents live here
│   │   └── opts.json
│   └── ...
│
├── containers/            # per-container runtime state
│   └── <container-id>/
│       ├── config.v2.json
│       ├── hostconfig.json
│       ├── <container-id>-json.log   # default logging driver writes here
│       └── ...
│
├── image/                 # image metadata, manifests, configs
├── network/               # network state
└── plugins/               # installed plugins
```

A few troubleshooting moves that use this knowledge:

- **Free up disk.** `docker system df -v` shows the breakdown. `du -sh /var/lib/docker/overlay2/* | sort -h | tail` reveals fat layers. `docker system prune` deletes unused.
- **Find a volume's contents from the host.** `docker volume inspect NAME --format '{{.Mountpoint}}'`, then `ls` that path.
- **Read a stopped container's logs without `docker logs`.** Tail `/var/lib/docker/containers/<id>/<id>-json.log` directly.

**On macOS / Windows** these paths are inside the Docker Desktop VM, not on your Mac/PC's filesystem. You can't `ls /var/lib/docker` from your shell — only inside containers. Docker Desktop's settings panel exposes the VM's disk usage in lieu of direct access.